In [1]:
!nvidia-smi | head -10
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

Sun May 17 15:36:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
torch=2.10.0+cu128  cuda=True  device=NVIDIA L4


In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


In [3]:
%%bash
# Lever A — batch-size sweep at fixed n=6, UTD=4
# 验证 SAC update 是否 launch-bound:若 mean_ms 随 batch 增大近似不变,
# 则可直接通过增大 batch 拿到大幅 wallclock 收益(零代码改动)。
mkdir -p experiments/profile
for bs in 256 512 1024 2048; do
  python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 6 --vector-mode async --device cuda \
    --updates-per-step 4 --batch-size $bs \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n6_utd4_bs${bs}.json
done

[profile] device=cuda  num_envs=6 (async)  obs_dim=40  action_dim=2  priv_dim=0  target=3300 transitions
[profile] warmup complete at 1500 transitions; measuring...
  device=cuda  num_envs=6 (async)  UTD=4  batch=256  asym=False  ln=False
  probe=s0  history=4  geom=upstream  tgt=1.5  obj=efficiency_v2
----------------------------------------------------------------------------------------
  measured 1800 transitions in 21.25s  ->  84.7 transitions/s   14.1 env.step/s   56.5 updates/s
----------------------------------------------------------------------------------------
  bucket             count    total_ms   mean_ms    p50_ms    p95_ms   % wallclock
  agent_act            300       266.7     0.889     0.883     0.927          1.3%
  env_step             300      7885.0    26.283    15.357    24.435         37.1%
  replay_add           300        20.2     0.067     0.063     0.081          0.1%
  replay_sample       1200       442.0     0.368     0.366     0.403          2.1%
  sac_

In [4]:
%%bash
# Lever B — n_envs sweep at fixed effective UTD ≈ 0.67
# effective UTD = updates_per_step / num_envs;保持与 thesis baseline (n=6, UTD=4)
# 等价的样本利用率,只看 wallclock 收益。L4 是 12 vCPU,推断 sweet spot 在 12-16。
# (n=6, UTD=4) baseline 已在 profile_train_env_completed.ipynb 跑过,这里不重复。

# n=12, UTD=8  → effective UTD = 8/12 ≈ 0.67
python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 12 --vector-mode async --device cuda \
    --updates-per-step 8 --batch-size 256 \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n12_utd8_bs256.json

# n=16, UTD=11 → effective UTD = 11/16 ≈ 0.69
python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 16 --vector-mode async --device cuda \
    --updates-per-step 11 --batch-size 256 \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n16_utd11_bs256.json

[profile] device=cuda  num_envs=12 (async)  obs_dim=40  action_dim=2  priv_dim=0  target=3300 transitions
[profile] warmup complete at 1500 transitions; measuring...
  device=cuda  num_envs=12 (async)  UTD=8  batch=256  asym=False  ln=False
  probe=s0  history=4  geom=upstream  tgt=1.5  obj=efficiency_v2
----------------------------------------------------------------------------------------
  measured 1800 transitions in 16.88s  ->  106.7 transitions/s   8.9 env.step/s   71.1 updates/s
----------------------------------------------------------------------------------------
  bucket             count    total_ms   mean_ms    p50_ms    p95_ms   % wallclock
  agent_act            150       137.5     0.917     0.912     0.945          0.8%
  env_step             150      3524.5    23.497    20.556    34.025         20.9%
  replay_add           150        15.6     0.104     0.101     0.153          0.1%
  replay_sample       1200       453.1     0.378     0.372     0.425          2.7%
  sa

In [5]:
%%bash
# Combo — Lever A + Lever B 同时打开
# n=12, UTD=8, batch=1024 → effective UTD ≈ 0.67,但每次 update 看到 4× 样本
# 这是 wallclock 最激进的免费午餐候选;若仍 ≤ 165 trans/s 说明假设不成立。
python -m scripts.profile_train \
    --flow wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy \
    --num-envs 12 --vector-mode async --device cuda \
    --updates-per-step 8 --batch-size 1024 \
    --measure-steps 1800 \
    --output-json experiments/profile/l4_n12_utd8_bs1024.json

[profile] device=cuda  num_envs=12 (async)  obs_dim=40  action_dim=2  priv_dim=0  target=3300 transitions
[profile] warmup complete at 1500 transitions; measuring...
  device=cuda  num_envs=12 (async)  UTD=8  batch=1024  asym=False  ln=False
  probe=s0  history=4  geom=upstream  tgt=1.5  obj=efficiency_v2
----------------------------------------------------------------------------------------
  measured 1800 transitions in 16.99s  ->  105.9 transitions/s   8.8 env.step/s   70.6 updates/s
----------------------------------------------------------------------------------------
  bucket             count    total_ms   mean_ms    p50_ms    p95_ms   % wallclock
  agent_act            150       134.8     0.898     0.896     0.934          0.8%
  env_step             150      3389.4    22.596    20.570    31.483         19.9%
  replay_add           150        15.2     0.102     0.099     0.118          0.1%
  replay_sample       1200       671.3     0.559     0.552     0.628          4.0%
  s